# 低評価予測モデルの比較と最終選択

## 目的

これまでに作成した特徴量を用いて、低評価予測モデルの最終的な特徴量構成を検討する。

ロジスティック回帰を基準モデルとし、その後Random ForestやBoosting系モデルと比較する。

## 評価方針

- Train：2017年12月まで
- Validation：2018年1月〜4月
- Holdout：2018年5月〜8月
- 特徴量選択・しきい値決定にはValidationを使用する
- Holdoutは最終モデル決定後に一度だけ評価する

## 評価指標

- F1
- Precision
- Recall
- ROC-AUC
- PR-AUC（Average Precision）

低評価レビュー（1〜2点）を陽性クラスとする。

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Notebookの作業ディレクトリを確認
current_dir = Path.cwd().resolve()

# プロジェクトルートを取得
if (current_dir / "src" / "project_paths.py").is_file():
    project_root = current_dir
elif (current_dir.parent / "src" / "project_paths.py").is_file():
    project_root = current_dir.parent
else:
    raise FileNotFoundError(
        "プロジェクトルートが見つかりません。"
        "Notebookの実行ディレクトリを確認してください。"
    )

# モジュール検索パスに追加
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 既存の共通パス設定を使用
from src.project_paths import DATA_DIR, PROJECT_ROOT

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)


PROJECT_ROOT: /workspaces/AWS_Python_data_analysis
DATA_DIR: /workspaces/AWS_Python_data_analysis/data


In [2]:
df_model = pd.read_csv(DATA_DIR / "model_improved_6.csv")

print("Shape:", df_model.shape)

print("\n=== 列一覧 ===")
print(df_model.columns.tolist())

print("\n=== order_idの重複数 ===")
print(df_model["order_id"].duplicated().sum())

print("\n=== 欠損数 ===")
print(df_model.isna().sum())

print("\n=== 購入月の範囲 ===")
print(df_model["purchase_month"].min())
print(df_model["purchase_month"].max())


Shape: (95824, 29)

=== 列一覧 ===
['order_id', 'review_score', 'total_price', 'total_freight', 'delivery_delay_days', 'purchase_month', 'customer_state', 'freight_ratio', 'purchase_month_num', 'item_count', 'category_count', 'low_review', 'payment_count', 'payment_type_count', 'main_payment_type', 'max_payment_installments', 'log_freight_ratio', 'month_sin', 'month_cos', 'main_category', 'main_category_gmv', 'customer_unique_id', 'prior_order_count', 'is_repeat_customer', 'prior_customer_gmv', 'prior_avg_order_value', 'days_since_previous_order', 'log_prior_customer_gmv', 'log_prior_avg_order_value']

=== order_idの重複数 ===
0

=== 欠損数 ===
order_id                     0
review_score                 0
total_price                  0
total_freight                0
delivery_delay_days          0
purchase_month               0
customer_state               0
freight_ratio                0
purchase_month_num           0
item_count                   0
category_count               0
low_review      

In [3]:
# Model A：注文・地域・購入時期
numeric_A = [
    "total_price",
    "total_freight",
    "delivery_delay_days",
    "log_freight_ratio",
    "item_count",
    "category_count",
    "month_sin",
    "month_cos",
]

categorical_A = [
    "customer_state",
]

# Model B：支払い特徴量を追加
numeric_B = numeric_A + [
    "payment_count",
    "payment_type_count",
    "max_payment_installments",
]

categorical_B = categorical_A + [
    "main_payment_type",
]

# Model C：商品カテゴリを追加
numeric_C = numeric_B.copy()

categorical_C = categorical_B + [
    "main_category",
]

# Model D：顧客履歴を追加
numeric_D = numeric_C + [
    "prior_order_count",
    "is_repeat_customer",
    "log_prior_customer_gmv",
    "log_prior_avg_order_value",
    "days_since_previous_order",
]

categorical_D = categorical_C.copy()

feature_sets = {
    "A_Order": (numeric_A, categorical_A),
    "B_Payment": (numeric_B, categorical_B),
    "C_Category": (numeric_C, categorical_C),
    "D_History": (numeric_D, categorical_D),
}

for name, (numeric, categorical) in feature_sets.items():
    print(
        name,
        "数値:",
        len(numeric),
        "カテゴリ:",
        len(categorical),
    )


A_Order 数値: 8 カテゴリ: 1
B_Payment 数値: 11 カテゴリ: 2
C_Category 数値: 11 カテゴリ: 3
D_History 数値: 16 カテゴリ: 3


In [4]:
# 時系列でTrain/Validationを分割

train = df_model[df_model["purchase_month"] <= "2017-12"].copy()

validation = df_model[
    (df_model["purchase_month"] >= "2018-01")
    & (df_model["purchase_month"] <= "2018-04")
].copy()

# Holdoutは分割件数のみ確認し、モデル比較では使用しない
holdout_count = (
    (df_model["purchase_month"] >= "2018-05")
    & (df_model["purchase_month"] <= "2018-08")
).sum()

print("Train:", len(train))
print("Validation:", len(validation))
print("Holdout（未使用）:", holdout_count)


Train: 43362
Validation: 27220
Holdout（未使用）: 25242


In [5]:
# モデルA-Dを同じ条件で評価する

from sklearn.metrics import average_precision_score, roc_auc_score

from src import model_metrics, model_training

results = []

for model_name, (numeric, categorical) in feature_sets.items():
    model, y_val, val_prob = model_training.train_logistic_model(
        train,
        validation,
        numeric,
        categorical,
    )

    threshold_results = model_metrics.threshold_table(
        y_val,
        val_prob,
        thresholds=np.arange(0.05, 0.51, 0.01),
    )

    best = threshold_results.sort_values(
        ["f1", "threshold"],
        ascending=[False, True],
    ).iloc[0]

    results.append(
        {
            "model": model_name,
            "threshold": best["threshold"],
            "precision": best["precision"],
            "recall": best["recall"],
            "f1": best["f1"],
            "roc_auc": roc_auc_score(y_val, val_prob),
            "pr_auc": average_precision_score(y_val, val_prob),
        }
    )

comparison = pd.DataFrame(results)

print(
    comparison.sort_values(
        "f1",
        ascending=False,
    ).to_string(index=False)
)

     model  threshold  precision   recall       f1  roc_auc   pr_auc
   A_Order       0.26   0.611806 0.502795 0.551970 0.766418 0.520047
 B_Payment       0.26   0.611656 0.502124 0.551504 0.767047 0.519549
C_Category       0.25   0.566923 0.510396 0.537176 0.770396 0.513088
 D_History       0.25   0.567756 0.508607 0.536557 0.770000 0.512457


## 時系列クロスバリデーション

学習期間を段階的に拡大するExpanding Window方式を採用し、
モデル評価の時期依存性と汎化性能を検証する。

2018年4月までのデータをfold1～fold4まで分解し、それぞれでTrain,Validationを作成する。2018年5〜8月のHoldoutは
このCVには使用しない。

この検討をする際に、レビューが確定していない注文が含まれている場合、その注文ラベルはその時点で利用できなかった可能性も考慮する必要がある。

In [6]:
# 時系列クロスバリデーションの期間設定
cv_periods = [
    {
        "fold": 1,
        "train_end": "2017-03",
        "val_start": "2017-04",
        "val_end": "2017-06",
    },
    {
        "fold": 2,
        "train_end": "2017-06",
        "val_start": "2017-07",
        "val_end": "2017-09",
    },
    {
        "fold": 3,
        "train_end": "2017-09",
        "val_start": "2017-10",
        "val_end": "2017-12",
    },
    {
        "fold": 4,
        "train_end": "2017-12",
        "val_start": "2018-01",
        "val_end": "2018-04",
    },
]

cv_splits = []
cv_summary = []

for period in cv_periods:
    train_idx = np.flatnonzero(
        train["purchase_month"].le(period["train_end"]).to_numpy()
    )

    val_idx = np.flatnonzero(
        train["purchase_month"]
        .between(period["val_start"], period["val_end"])
        .to_numpy()
    )

    # 学習・検証データが存在することを確認
    assert len(train_idx) > 0
    assert len(val_idx) > 0

    # 同じ注文が学習と検証に含まれていないことを確認
    assert len(set(train_idx) & set(val_idx)) == 0

    # 学習期間が検証期間より過去にあることを確認
    assert period["train_end"] < period["val_start"]

    cv_splits.append((train_idx, val_idx))

    cv_summary.append(
        {
            "fold": period["fold"],
            "train_end": period["train_end"],
            "val_start": period["val_start"],
            "val_end": period["val_end"],
            "train_count": len(train_idx),
            "val_count": len(val_idx),
            "train_low_review_rate": train.iloc[train_idx]["low_review"].mean(),
            "val_low_review_rate": train.iloc[val_idx]["low_review"].mean(),
        }
    )

df_cv_summary = pd.DataFrame(cv_summary)

display(df_cv_summary)


AssertionError: 

In [ ]:
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from src import model_training

cv_results = []

for fold, (train_idx, val_idx) in enumerate(cv_splits, start=1):
    fold_train = train.iloc[train_idx].copy()
    fold_val = train.iloc[val_idx].copy()

    model, y_val, val_prob = model_training.train_logistic_model(
        fold_train,
        fold_val,
        numeric_A,
        categorical_A,
    )

    # 前回のValidationで選択したModel Aのしきい値
    threshold = 0.26
    val_pred = (val_prob >= threshold).astype(int)

    cv_results.append(
        {
            "fold": fold,
            "train_count": len(fold_train),
            "val_count": len(fold_val),
            "precision": precision_score(y_val, val_pred, zero_division=0),
            "recall": recall_score(y_val, val_pred, zero_division=0),
            "f1": f1_score(y_val, val_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_val, val_prob),
            "ap": average_precision_score(y_val, val_prob),
        }
    )

df_cv_results = pd.DataFrame(cv_results)

display(df_cv_results)

print("\n=== CV平均 ===")
print(df_cv_results[["precision", "recall", "f1", "roc_auc", "ap"]].mean().round(4))

print("\n=== CV標準偏差 ===")
print(df_cv_results[["precision", "recall", "f1", "roc_auc", "ap"]].std().round(4))


,fold,train_count,val_count,precision,recall,f1,roc_auc,ap
0,1,5175,8918,0.304569,0.061538,0.102389,0.627780,0.195800
1,2,14093,12125,0.219293,0.307047,0.255855,0.635026,0.213342
2,3,26218,17144,0.621200,0.322281,0.424388,0.740900,0.431479



=== CV平均 ===
precision    0.3817
recall       0.2303
f1           0.2609
roc_auc      0.6679
ap           0.2802
dtype: float64

=== CV標準偏差 ===
precision    0.2118
recall       0.1463
f1           0.1611
roc_auc      0.0633
ap           0.1313
dtype: float64


In [ ]:
from src import model_metrics

threshold_diagnostics = []

for fold, (train_idx, val_idx) in enumerate(cv_splits, start=1):
    fold_train = train.iloc[train_idx].copy()
    fold_val = train.iloc[val_idx].copy()

    model, y_val, val_prob = model_training.train_logistic_model(
        fold_train,
        fold_val,
        numeric_A,
        categorical_A,
    )

    threshold_results = model_metrics.threshold_table(
        y_val,
        val_prob,
        thresholds=np.arange(0.05, 0.51, 0.01),
    )

    best = threshold_results.sort_values(
        ["f1", "threshold"],
        ascending=[False, True],
    ).iloc[0]

    threshold_diagnostics.append(
        {
            "fold": fold,
            "best_threshold_diagnostic": best["threshold"],
            "best_f1_diagnostic": best["f1"],
            "f1_at_026": df_cv_results.loc[df_cv_results["fold"] == fold, "f1"].iloc[0],
        }
    )

df_threshold_diagnostics = pd.DataFrame(threshold_diagnostics)

display(df_threshold_diagnostics)


,fold,best_threshold_diagnostic,best_f1_diagnostic,f1_at_026
0,1,0.14,0.263661,0.102389
1,2,0.28,0.261934,0.255855
2,3,0.21,0.479328,0.424388


In [ ]:
# 購入日時とレビュー投稿日時が異なることを踏まえた調整
from src.project_paths import DATA_DIR

df_review_anchor = pd.read_csv(DATA_DIR / "review_anchor.csv")

print("=== Shape ===")
print(df_review_anchor.shape)

print("\n=== 列一覧 ===")
print(df_review_anchor.columns.tolist())

print("\n=== 先頭5行 ===")
display(df_review_anchor.head())

print("\n=== order_id重複数 ===")
print(df_review_anchor["order_id"].duplicated().sum())

print("\n=== 欠損数 ===")
print(df_review_anchor.isna().sum())

print("\n=== モデルデータとのorder_id一致数 ===")
print(df_model["order_id"].isin(df_review_anchor["order_id"]).sum())

=== Shape ===
(94865, 4)

=== 列一覧 ===
['order_id', 'review_score', 'review_timestamp', 'low_review']

=== 先頭5行 ===


,order_id,review_score,review_timestamp,low_review
0,00010242fe8c5a6d1ba2dd792cb16214,5,2017-09-22 10:57:03.000,0
1,00018f77f2f0320c557190d7a144bdd3,4,2017-05-15 11:34:13.000,0
2,000229ec398224ef6ca0657da4fc703e,5,2018-01-23 16:06:31.000,0
3,00024acbcdf0a6daa1e931b038114c75,4,2018-08-15 16:39:01.000,0
4,00042b26cf59d7ce69dfabb4e55b4fd9,5,2017-03-03 10:54:59.000,0



=== order_id重複数 ===
0

=== 欠損数 ===
order_id            0
review_score        0
review_timestamp    0
low_review          0
dtype: int64

=== モデルデータとのorder_id一致数 ===
92211


In [ ]:
# レビュー日時データとの一致状況を確認
df_anchor_check = df_model[
    ["order_id", "review_score", "low_review", "purchase_month"]
].merge(
    df_review_anchor[
        ["order_id", "review_score", "low_review", "review_timestamp"]
    ].rename(
        columns={
            "review_score": "anchor_review_score",
            "low_review": "anchor_low_review",
        }
    ),
    on="order_id",
    how="left",
    validate="one_to_one",
    indicator=True,
)

# 一致状況
print("=== 一致状況 ===")
print(df_anchor_check["_merge"].value_counts())

# 一致・不一致別の低評価率
print("\n=== 低評価率 ===")
display(
    df_anchor_check.groupby("_merge", observed=True).agg(
        order_count=("order_id", "size"),
        low_review_rate=("low_review", "mean"),
    )
)

# 一致した注文でラベルが一致するか
matched = df_anchor_check[df_anchor_check["_merge"] == "both"].copy()

print("\n=== レビュースコア不一致件数 ===")
print((matched["review_score"] != matched["anchor_review_score"]).sum())

print("\n=== 低評価ラベル不一致件数 ===")
print((matched["low_review"] != matched["anchor_low_review"]).sum())

# レビュー日時が購入月より前になっていないか
matched["review_timestamp"] = pd.to_datetime(
    matched["review_timestamp"],
    errors="coerce",
)

matched["purchase_month_start"] = pd.to_datetime(matched["purchase_month"] + "-01")

print("\n=== レビュー日時の変換失敗件数 ===")
print(matched["review_timestamp"].isna().sum())

print("\n=== レビュー日時が購入月初日より前の件数 ===")
print((matched["review_timestamp"] < matched["purchase_month_start"]).sum())


=== 一致状況 ===
_merge
both          92211
left_only      3613
right_only        0
Name: count, dtype: int64

=== 低評価率 ===


,order_count,low_review_rate
_merge,,
left_only,3613,0.290064
both,92211,0.121330



=== レビュースコア不一致件数 ===
141

=== 低評価ラベル不一致件数 ===
35

=== レビュー日時の変換失敗件数 ===
0

=== レビュー日時が購入月初日より前の件数 ===
3


In [ ]:
from pathlib import Path

import pandas as pd

from src.project_paths import DATA_DIR

# 既存CSVの中から、必要な日時列を探す
date_keywords = [
    "purchase",
    "delivered",
    "delivery",
    "review",
    "timestamp",
    "date",
]

for path in sorted(DATA_DIR.glob("*.csv")):
    columns = pd.read_csv(path, nrows=0).columns.tolist()

    date_columns = [
        col
        for col in columns
        if any(keyword in col.lower() for keyword in date_keywords)
    ]

    if date_columns:
        print(f"\n=== {path.name} ===")
        print(date_columns)



=== delivery_delay_review_summary.csv ===
['review_score', 'delivery_delay_days']

=== future_90d_purchase_features.csv ===
['review_score', 'low_review', 'review_timestamp']

=== model_improved_6.csv ===
['review_score', 'delivery_delay_days', 'purchase_month', 'purchase_month_num', 'low_review']

=== monthly_category_sales_by_state.csv ===
['purchase_month']

=== order_level_sales_by_state.csv ===
['purchase_month']

=== repeat_analysis_adjustments.csv ===
['purchase_timestamp']

=== review_anchor.csv ===
['review_score', 'review_timestamp', 'low_review']

=== review_order_summary.csv ===
['review_score']

=== sales_baseline_by_month_state.csv ===
['purchase_month']


In [ ]:
from src.project_paths import DATA_DIR

df_order_dates = pd.read_csv(
    DATA_DIR / "order_delivery_timestamps.csv",
    parse_dates=["purchase_timestamp", "delivered_timestamp"],
)

df_delivery_check = df_model[["order_id", "purchase_month", "low_review"]].merge(
    df_order_dates,
    on="order_id",
    how="left",
    validate="one_to_one",
)

# 配送完了月
df_delivery_check["delivered_month"] = (
    df_delivery_check["delivered_timestamp"].dt.to_period("M").astype("string")
)

print("=== モデルデータとの結合 ===")
print("行数:", len(df_delivery_check))
print("購入日時の欠損:", df_delivery_check["purchase_timestamp"].isna().sum())
print("配送完了日時の欠損:", df_delivery_check["delivered_timestamp"].isna().sum())

print("\n=== 購入月と配送完了月の比較 ===")
has_delivery = df_delivery_check["delivered_timestamp"].notna()

print(
    "購入月と配送完了月が異なる注文数:",
    (
        df_delivery_check.loc[has_delivery, "purchase_month"]
        != df_delivery_check.loc[has_delivery, "delivered_month"]
    ).sum(),
)

print("\n=== 配送完了日時の有無による低評価率 ===")
display(
    df_delivery_check.assign(has_delivery=has_delivery)
    .groupby("has_delivery")
    .agg(
        order_count=("order_id", "size"),
        low_review_rate=("low_review", "mean"),
    )
)

print("\n=== 配送完了日時が欠損している注文のステータス ===")
display(df_delivery_check.loc[~has_delivery, "order_status"].value_counts(dropna=False))


=== モデルデータとの結合 ===
行数: 95824
購入日時の欠損: 0
配送完了日時の欠損: 0

=== 購入月と配送完了月の比較 ===
購入月と配送完了月が異なる注文数: 36091

=== 配送完了日時の有無による低評価率 ===


,order_count,low_review_rate
has_delivery,,
True,95824,0.127692



=== 配送完了日時が欠損している注文のステータス ===


Series([], Name: count, dtype: int64)

購入月と配送完了月が異なる件数が約37.7%存在するので、「配送完了後、低評価を予測するモデル」を作成するにあたって、これまでの購入月に基づくCVと、配送完了月に基づくCVでは、学習・検証対象が大きく変わる可能性がある。

また、配送完了日時より前に、レビューが回答されている可能性がないかどうかを確かめる必要がある。

In [ ]:
# 配送完了日時を基準にした月別件数
df_delivery_month_summary = (
    df_delivery_check.groupby("delivered_month", dropna=False)
    .agg(
        order_count=("order_id", "size"),
        low_review_rate=("low_review", "mean"),
    )
    .reset_index()
    .sort_values("delivered_month")
)

print("=== 配送完了月別の注文件数と低評価率 ===")
display(df_delivery_month_summary)

print("\n=== 配送完了月の範囲 ===")
print("最小:", df_delivery_check["delivered_timestamp"].min())
print("最大:", df_delivery_check["delivered_timestamp"].max())

print("\n=== 2018年9月以降に配送完了した注文 ===")
after_august = df_delivery_check["delivered_timestamp"] >= pd.Timestamp("2018-09-01")

print("件数:", after_august.sum())
print("低評価率:", df_delivery_check.loc[after_august, "low_review"].mean())

print("\n=== 購入から配送完了までの日数 ===")
delivery_days = (
    df_delivery_check["delivered_timestamp"] - df_delivery_check["purchase_timestamp"]
).dt.total_seconds() / 86400

display(delivery_days.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))


=== 配送完了月別の注文件数と低評価率 ===


,delivered_month,order_count,low_review_rate
0,2016-10,204,0.142157
1,2016-11,57,0.245614
2,2016-12,3,0.333333
3,2017-01,281,0.085409
4,2017-02,1344,0.091518
5,2017-03,2363,0.098603
6,2017-04,1836,0.122549
7,2017-05,3731,0.118199
8,2017-06,3194,0.103006
9,2017-07,3425,0.097518



=== 配送完了月の範囲 ===
最小: 2016-10-11 13:46:32
最大: 2018-10-17 13:22:46

=== 2018年9月以降に配送完了した注文 ===
件数: 58
低評価率: 0.6724137931034483

=== 購入から配送完了までの日数 ===


count    95824.000000
mean        12.516765
std          9.460505
min          0.533414
50%         10.206059
90%         23.030749
95%         29.130834
99%         45.852779
max        208.351759
dtype: float64

In [ ]:
# レビュー日時の確認
df_review_timing = df_delivery_check[
    ["order_id", "purchase_timestamp", "delivered_timestamp", "low_review"]
].merge(
    df_review_anchor[["order_id", "review_timestamp"]],
    on="order_id",
    how="left",
    validate="one_to_one",
)

df_review_timing["review_timestamp"] = pd.to_datetime(
    df_review_timing["review_timestamp"],
    errors="coerce",
)

has_review_time = df_review_timing["review_timestamp"].notna()

# レビュー回答日時と配送完了日時の比較
review_before_delivery = has_review_time & (
    df_review_timing["review_timestamp"] < df_review_timing["delivered_timestamp"]
)

review_after_delivery = has_review_time & (
    df_review_timing["review_timestamp"] >= df_review_timing["delivered_timestamp"]
)

print("=== レビュー日時の確認 ===")
print("モデルデータ:", len(df_review_timing))
print("レビュー日時あり:", has_review_time.sum())
print("レビュー日時なし:", (~has_review_time).sum())

print("\n=== レビューと配送完了の前後関係 ===")
print("配送完了前にレビュー回答:", review_before_delivery.sum())
print("配送完了後にレビュー回答:", review_after_delivery.sum())

print("\n=== 配送完了前にレビューされた注文の低評価率 ===")
print(df_review_timing.loc[review_before_delivery, "low_review"].mean())

print("\n=== 配送完了からレビュー回答までの日数 ===")
days_to_review = (
    df_review_timing.loc[review_after_delivery, "review_timestamp"]
    - df_review_timing.loc[review_after_delivery, "delivered_timestamp"]
).dt.total_seconds() / 86400

display(days_to_review.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))


=== レビュー日時の確認 ===
モデルデータ: 95824
レビュー日時あり: 92211
レビュー日時なし: 3613

=== レビューと配送完了の前後関係 ===
配送完了前にレビュー回答: 4502
配送完了後にレビュー回答: 87709

=== 配送完了前にレビューされた注文の低評価率 ===
0.760772989782319

=== 配送完了からレビュー回答までの日数 ===


count    87709.000000
mean         3.569379
std         10.089106
min          0.000926
50%          2.017512
90%          5.439076
95%          7.951076
99%         24.060826
max        518.979074
dtype: float64

In [7]:
import pandas as pd

from src.project_paths import DATA_DIR

# STEP 50-3：レビュー日時の分類データを読み込む
df_review_timing_features = pd.read_csv(
    DATA_DIR / "review_timing_features.csv",
    parse_dates=[
        "delivered_timestamp",
        "first_review_timestamp",
        "last_review_timestamp",
    ],
)

print("=== レビュー日時データ ===")
print("行数:", len(df_review_timing_features))
print(
    "order_id重複:",
    df_review_timing_features["order_id"].duplicated().sum(),
)

# モデルデータと結合
df_timing_model = df_model[["order_id", "low_review", "review_score"]].merge(
    df_review_timing_features,
    on="order_id",
    how="left",
    validate="one_to_one",
    indicator=True,
)

print("\n=== モデルデータとの結合 ===")
print("結合後の行数:", len(df_timing_model))
print(df_timing_model["_merge"].value_counts())

print("\n=== レビュー日時分類別の注文件数・低評価率 ===")
timing_summary = (
    df_timing_model.groupby("timing_category", dropna=False)
    .agg(
        order_count=("order_id", "size"),
        low_review_count=("low_review", "sum"),
        low_review_rate=("low_review", "mean"),
        multiple_review_orders=(
            "review_count",
            lambda x: x.gt(1).sum(),
        ),
    )
    .reset_index()
    .sort_values("order_count", ascending=False)
)

display(timing_summary)

print("\n=== 分類別の全体比率 ===")
timing_summary["share"] = timing_summary["order_count"] / len(df_timing_model)

display(timing_summary[["timing_category", "order_count", "share"]])

print("\n=== レビュー日時の欠損 ===")
print(
    "最初のレビュー日時なし:",
    df_timing_model["first_review_timestamp"].isna().sum(),
)
print(
    "最後のレビュー日時なし:",
    df_timing_model["last_review_timestamp"].isna().sum(),
)


=== レビュー日時データ ===
行数: 98673
order_id重複: 0

=== モデルデータとの結合 ===
結合後の行数: 95824
_merge
both          95824
left_only         0
right_only        0
Name: count, dtype: int64

=== レビュー日時分類別の注文件数・低評価率 ===


,timing_category,order_count,low_review_count,low_review_rate,multiple_review_orders
0,all_reviews_after_delivery,87683,7758,0.088478,355
1,all_reviews_before_delivery,4390,3417,0.778360,23
2,review_timestamp_missing,3646,1055,0.289358,42
3,reviews_before_and_after_delivery,105,6,0.057143,105



=== 分類別の全体比率 ===


,timing_category,order_count,share
0,all_reviews_after_delivery,87683,0.915042
1,all_reviews_before_delivery,4390,0.045813
2,review_timestamp_missing,3646,0.038049
3,reviews_before_and_after_delivery,105,0.001096



=== レビュー日時の欠損 ===
最初のレビュー日時なし: 3613
最後のレビュー日時なし: 3613


In [8]:
print("=== レビュー件数の分布 ===")

review_count_summary = (
    df_timing_model.groupby("review_count", dropna=False)
    .agg(
        order_count=("order_id", "size"),
        low_review_rate=("low_review", "mean"),
    )
    .reset_index()
    .sort_values("review_count")
)

display(review_count_summary)

print("\n=== 複数レビュー注文の分類 ===")

multiple_review_summary = (
    df_timing_model.loc[df_timing_model["review_count"] > 1]
    .groupby("timing_category", dropna=False)
    .agg(
        order_count=("order_id", "size"),
        low_review_rate=("low_review", "mean"),
    )
    .reset_index()
)

display(multiple_review_summary)


=== レビュー件数の分布 ===


,review_count,order_count,low_review_rate
0,1,95299,0.127903
1,2,521,0.088292
2,3,4,0.250000



=== 複数レビュー注文の分類 ===


,timing_category,order_count,low_review_rate
0,all_reviews_after_delivery,355,0.059155
1,all_reviews_before_delivery,23,0.565217
2,review_timestamp_missing,42,0.166667
3,reviews_before_and_after_delivery,105,0.057143


In [9]:
# STEP 51-1：レビュー日時の欠損状態を詳しく確認

df_timing_model["missing_review_timestamp_count"] = df_timing_model[
    "missing_review_timestamp_count"
].astype("int64")

missing_timing = df_timing_model.loc[
    df_timing_model["timing_category"] == "review_timestamp_missing"
].copy()

print("=== レビュー日時欠損の内訳 ===")

display(
    missing_timing.groupby(["review_count", "timestamp_count"], dropna=False)
    .agg(
        order_count=("order_id", "size"),
        low_review_rate=("low_review", "mean"),
    )
    .reset_index()
    .sort_values(["review_count", "timestamp_count"])
)

print("\n=== 日時が一部だけ存在する注文 ===")

partial_timing = missing_timing.loc[missing_timing["timestamp_count"] > 0]

print("件数:", len(partial_timing))

display(
    partial_timing[
        [
            "order_id",
            "review_count",
            "timestamp_count",
            "first_review_timestamp",
            "last_review_timestamp",
            "delivered_timestamp",
            "low_review",
        ]
    ].head(10)
)

print("\n=== 暫定予測対象の確認 ===")

df_timing_model["prediction_eligible_candidate"] = (
    df_timing_model["timing_category"] == "all_reviews_after_delivery"
)

display(
    df_timing_model.groupby("prediction_eligible_candidate").agg(
        order_count=("order_id", "size"),
        low_review_count=("low_review", "sum"),
        low_review_rate=("low_review", "mean"),
    )
)


=== レビュー日時欠損の内訳 ===


,review_count,timestamp_count,order_count,low_review_rate
0,1,0,3604,0.290788
1,2,0,9,0.000000
2,2,1,32,0.218750
3,3,2,1,0.000000



=== 日時が一部だけ存在する注文 ===
件数: 33


,order_id,review_count,timestamp_count,first_review_timestamp,last_review_timestamp,delivered_timestamp,low_review
1888,34de5f7952fbd04a9a1e5d7001b014b5,2,1,2018-03-10 00:46:40,2018-03-10 00:46:40,2018-03-08 21:18:31,1
4885,8681822ccf2aaea9124f49a2a6dba7b1,2,1,2018-03-26 21:15:33,2018-03-26 21:15:33,2018-03-14 19:58:50,1
9401,075a544c5f4ed4bb75f82b160465fe76,2,1,2018-03-07 12:32:38,2018-03-07 12:32:38,2018-03-05 20:58:53,1
9802,29f5dd847b7db645b7ce75cc49ddb612,2,1,2018-05-27 14:26:51,2018-05-27 14:26:51,2018-05-07 19:36:52,1
10448,6401cfe160d72392195bd6e311a582cd,2,1,2018-03-22 21:07:04,2018-03-22 21:07:04,2018-03-24 17:02:43,1
11240,ab71e5dff0ed652f1559e1860391bd22,2,1,2018-03-07 12:32:38,2018-03-07 12:32:38,2018-03-22 16:27:35,1
12173,f9fe6e11e33871e71f921d5051e72bb4,2,1,2018-03-19 11:35:24,2018-03-19 11:35:24,2018-03-17 17:08:21,1
12353,03c939fd7fd3b38f8485a0f95798f1f6,3,2,2018-03-06 19:50:32,2018-03-30 00:29:09,2018-03-19 21:48:52,0
13497,29062384ce4975f78aeba6a496510386,2,1,2018-03-14 18:21:23,2018-03-14 18:21:23,2018-03-14 16:58:50,0
13665,2e84df8fa5a2f0d5eab815d6c157b1f3,2,1,2017-10-10 22:47:23,2017-10-10 22:47:23,2017-10-24 10:38:37,0



=== 暫定予測対象の確認 ===


,order_count,low_review_count,low_review_rate
prediction_eligible_candidate,,,
False,8141,4478,0.550055
True,87683,7758,0.088478


In [10]:
# STEP 51-2：各Foldの学習開始時点で利用可能なラベルを確認

fold_boundaries = [
    ("Fold 1", "2017-04-01"),
    ("Fold 2", "2017-07-01"),
    ("Fold 3", "2017-10-01"),
    ("Fold 4", "2018-01-01"),
]

df_eligible = df_timing_model.loc[
    df_timing_model["prediction_eligible_candidate"]
].copy()

results = []

for fold_name, boundary_str in fold_boundaries:
    boundary = pd.Timestamp(boundary_str)

    # 分割境界より前に配送完了した注文
    train_candidates = df_eligible.loc[df_eligible["delivered_timestamp"] < boundary]

    # 分割境界までに全レビューの回答が完了していた注文
    label_available = train_candidates["last_review_timestamp"] < boundary

    results.append(
        {
            "fold": fold_name,
            "training_boundary": boundary_str,
            "candidate_orders": len(train_candidates),
            "label_available_orders": int(label_available.sum()),
            "label_not_yet_available": int((~label_available).sum()),
            "label_available_rate": (
                label_available.mean() if len(train_candidates) > 0 else float("nan")
            ),
            "low_review_rate_available": (
                train_candidates.loc[label_available, "low_review"].mean()
            ),
        }
    )

df_label_availability = pd.DataFrame(results)

print("=== 学習境界時点のラベル利用可能性 ===")
display(df_label_availability)


=== 学習境界時点のラベル利用可能性 ===


,fold,training_boundary,candidate_orders,label_available_orders,label_not_yet_available,label_available_rate,low_review_rate_available
0,Fold 1,2017-04-01,4025,3711,314,0.921988,0.089733
1,Fold 2,2017-07-01,12212,11802,410,0.966426,0.091002
2,Fold 3,2017-10-01,23136,22667,469,0.979729,0.085013
3,Fold 4,2018-01-01,38127,37537,590,0.984525,0.086635


In [11]:
import pandas as pd

# 配送完了後に全レビューが回答された注文
df_cv_base = df_timing_model.loc[
    df_timing_model["timing_category"] == "all_reviews_after_delivery"
].copy()

# 配送完了日時・最終レビュー回答日時があることを確認
assert df_cv_base["delivered_timestamp"].notna().all()
assert df_cv_base["last_review_timestamp"].notna().all()
assert df_cv_base["order_id"].is_unique

fold_configs = [
    ("Fold 1", "2017-04-01", "2017-07-01"),
    ("Fold 2", "2017-07-01", "2017-10-01"),
    ("Fold 3", "2017-10-01", "2018-01-01"),
    ("Fold 4", "2018-01-01", "2018-05-01"),
]

fold_splits = {}
fold_results = []

for fold_name, val_start_str, val_end_str in fold_configs:
    val_start = pd.Timestamp(val_start_str)
    val_end = pd.Timestamp(val_end_str)

    # 学習：検証開始前に配送され、ラベルも確定している注文
    train_mask = (df_cv_base["delivered_timestamp"] < val_start) & (
        df_cv_base["last_review_timestamp"] < val_start
    )

    # 検証：配送完了日時が検証期間内の注文
    val_mask = (df_cv_base["delivered_timestamp"] >= val_start) & (
        df_cv_base["delivered_timestamp"] < val_end
    )

    df_train = df_cv_base.loc[train_mask].copy()
    df_val = df_cv_base.loc[val_mask].copy()

    assert set(df_train["order_id"]).isdisjoint(set(df_val["order_id"]))

    fold_splits[fold_name] = {
        "train_ids": df_train["order_id"].tolist(),
        "val_ids": df_val["order_id"].tolist(),
    }

    fold_results.append(
        {
            "fold": fold_name,
            "train_until": val_start_str,
            "validation_period": (f"{val_start_str} ～ {val_end_str}"),
            "train_orders": len(df_train),
            "val_orders": len(df_val),
            "train_low_review_rate": df_train["low_review"].mean(),
            "val_low_review_rate": df_val["low_review"].mean(),
            "train_max_delivered": df_train["delivered_timestamp"].max(),
            "train_max_review": df_train["last_review_timestamp"].max(),
            "val_min_delivered": df_val["delivered_timestamp"].min(),
            "val_max_delivered": df_val["delivered_timestamp"].max(),
        }
    )

df_fold_summary = pd.DataFrame(fold_results)

print("=== 配送完了日時基準・4-Fold CV ===")
display(df_fold_summary)


=== 配送完了日時基準・4-Fold CV ===


,fold,train_until,validation_period,train_orders,val_orders,train_low_review_rate,val_low_review_rate,train_max_delivered,train_max_review,val_min_delivered,val_max_delivered
0,Fold 1,2017-04-01,2017-04-01 ～ 2017-07-01,3711,8187,0.089733,0.091242,2017-03-30 14:05:20,2017-03-31 23:32:31,2017-04-01 07:38:01,2017-06-30 19:09:42
1,Fold 2,2017-07-01,2017-07-01 ～ 2017-10-01,11802,10924,0.091002,0.079000,2017-06-29 16:32:57,2017-06-30 23:58:51,2017-07-01 07:36:19,2017-09-30 20:09:53
2,Fold 3,2017-10-01,2017-10-01 ～ 2018-01-01,22667,14991,0.085013,0.089320,2017-09-29 20:19:57,2017-09-30 23:59:53,2017-10-01 11:52:06,2017-12-31 15:19:16
3,Fold 4,2018-01-01,2018-01-01 ～ 2018-05-01,37537,23557,0.086635,0.100777,2017-12-30 18:41:57,2017-12-31 23:27:08,2018-01-01 05:47:11,2018-04-30 23:58:51


In [18]:
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

from src.model_metrics import threshold_metrics
from src.model_training import train_logistic_model

# Model Aの特徴量
numeric_features_a = [
    "total_price",
    "total_freight",
    "delivery_delay_days",
    "log_freight_ratio",
    "item_count",
    "category_count",
    "month_sin",
    "month_cos",
]

categorical_features_a = ["customer_state"]

# モデルデータと配送・レビュー日時を結合
df_cv_model = df_model.merge(
    df_cv_base[["order_id", "delivered_timestamp", "last_review_timestamp"]],
    on="order_id",
    how="inner",
    validate="one_to_one",
)

assert len(df_cv_model) == 87683

cv_predictions = []
cv_results = []

for fold_name, start, end in fold_configs:
    start, end = pd.Timestamp(start), pd.Timestamp(end)

    train = df_cv_model.loc[
        (df_cv_model["delivered_timestamp"] < start)
        & (df_cv_model["last_review_timestamp"] < start)
    ]

    val = df_cv_model.loc[
        (df_cv_model["delivered_timestamp"] >= start)
        & (df_cv_model["delivered_timestamp"] < end)
    ]

    model, y_val, probabilities = train_logistic_model(
        train,
        val,
        numeric_features_a,
        categorical_features_a,
    )

    cv_predictions.append(
        pd.DataFrame(
            {
                "fold": fold_name,
                "order_id": val["order_id"].to_numpy(),
                "low_review": y_val.to_numpy(),
                "probability": probabilities,
            }
        )
    )

    metrics = threshold_metrics(y_val, probabilities, threshold=0.24)

    cv_results.append(
        {
            "fold": fold_name,
            "train_orders": len(train),
            "val_orders": len(val),
            "val_low_review_rate": y_val.mean(),
            "roc_auc": roc_auc_score(y_val, probabilities),
            "average_precision": average_precision_score(y_val, probabilities),
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "f1": metrics["f1"],
        }
    )

df_cv_results = pd.DataFrame(cv_results)

print("=== Model A：配送完了日時基準4-Fold CV ===")
display(df_cv_results)

print("\n=== 4-Foldの平均・標準偏差 ===")
display(
    df_cv_results[
        [
            "roc_auc",
            "average_precision",
            "precision",
            "recall",
            "f1",
        ]
    ].agg(["mean", "std"])
)


=== Model A：配送完了日時基準4-Fold CV ===


,fold,train_orders,val_orders,val_low_review_rate,roc_auc,average_precision,precision,recall,f1
0,Fold 1,3711,8187,0.091242,0.578630,0.132041,0.309524,0.017403,0.032953
1,Fold 2,11802,10924,0.079000,0.562999,0.112763,0.155203,0.101970,0.123077
2,Fold 3,22667,14991,0.089320,0.630262,0.183784,0.372385,0.066468,0.112801
3,Fold 4,37537,23557,0.100777,0.643905,0.212996,0.413551,0.074558,0.126338



=== 4-Foldの平均・標準偏差 ===


,roc_auc,average_precision,precision,recall,f1
mean,0.603949,0.160396,0.312666,0.065100,0.098792
std,0.039187,0.046140,0.113356,0.035241,0.044270


In [17]:
from src.model_metrics import threshold_table

# 各Foldの予測結果を結合
df_cv_predictions = pd.concat(
    cv_predictions,
    ignore_index=True,
)

# 過去の基準しきい値
baseline_threshold = 0.24

# Foldごとにしきい値を比較
for fold_name, group in df_cv_predictions.groupby("fold", sort=True):
    threshold_results = threshold_table(
        group["low_review"],
        group["probability"],
    )

    # F1が高い上位10件のみ表示
    top10 = threshold_results.sort_values(
        ["f1", "threshold"],
        ascending=[False, True],
    ).head(10)

    print(f"\n=== {fold_name}：F1上位10件 ===")
    display(top10)



=== Fold 1：F1上位10件 ===


,threshold,accuracy,precision,recall,f1
6,0.07,0.756687,0.138292,0.318608,0.192869
5,0.06,0.615977,0.114754,0.477912,0.185070
4,0.05,0.417369,0.103645,0.704150,0.180694
7,0.08,0.823256,0.151394,0.203481,0.173615
3,0.04,0.237327,0.095868,0.872825,0.172761
2,0.03,0.151215,0.092510,0.942436,0.168482
1,0.02,0.110419,0.091704,0.982597,0.167752
0,0.01,0.092342,0.091243,0.998661,0.167208
8,0.09,0.859045,0.174400,0.145917,0.158892
9,0.10,0.875290,0.172249,0.096386,0.123605



=== Fold 2：F1上位10件 ===


,threshold,accuracy,precision,recall,f1
13,0.14,0.508696,0.093061,0.596756,0.161013
14,0.15,0.581838,0.094550,0.500579,0.159057
16,0.17,0.701117,0.102054,0.356895,0.158722
15,0.16,0.646558,0.097692,0.421784,0.158640
17,0.18,0.751465,0.107959,0.295481,0.158140
12,0.13,0.426034,0.088946,0.677868,0.157258
19,0.20,0.825613,0.126791,0.205098,0.156707
18,0.19,0.792384,0.115068,0.243337,0.156250
11,0.12,0.337605,0.084388,0.749710,0.151700
10,0.11,0.251556,0.082734,0.840093,0.150634



=== Fold 3：F1上位10件 ===


,threshold,accuracy,precision,recall,f1
11,0.12,0.847175,0.220985,0.281553,0.247619
10,0.11,0.804216,0.183584,0.345780,0.239834
12,0.13,0.870522,0.251240,0.227035,0.238525
13,0.14,0.885998,0.289294,0.189694,0.229138
9,0.10,0.733974,0.154268,0.441374,0.228627
8,0.09,0.611367,0.127387,0.572816,0.208424
14,0.15,0.893536,0.309064,0.155340,0.206759
15,0.16,0.897605,0.325000,0.135922,0.191680
7,0.08,0.389967,0.104158,0.766990,0.183409
16,0.17,0.900540,0.341667,0.122479,0.180319



=== Fold 4：F1上位10件 ===


,threshold,accuracy,precision,recall,f1
11,0.12,0.837713,0.249221,0.303286,0.273608
10,0.11,0.798489,0.212642,0.369840,0.270029
12,0.13,0.860678,0.283603,0.250632,0.266100
9,0.10,0.726069,0.176013,0.466723,0.255623
13,0.14,0.876640,0.320027,0.199242,0.245587
8,0.09,0.578894,0.140256,0.619629,0.228736
14,0.15,0.884153,0.346054,0.168071,0.226255
15,0.16,0.888611,0.368697,0.147852,0.211064
16,0.17,0.891073,0.384892,0.135215,0.200125
7,0.08,0.316212,0.112690,0.841618,0.198766
